# Notebook 5: Failure Casebook and Qualitative Sampling

**Goal:** Prepare a dataset for later open coding.

**Inputs:** `failure_cases.csv`, `failure_cases.jsonl`

**RQs addressed:** RQ8

Run the failure export first::

    python -m analysis export-failures --results Output/**/*.csv --out Analysis/output/failures --markdown

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from analysis.plots import failure_category_bar
from analysis.schema import PRELIMINARY_FAILURE_LABELS

sns.set_theme(style='whitegrid')

DATA_DIR = Path('../output/failures')
cases_path = DATA_DIR / 'failure_cases.csv'

if cases_path.exists():
    cases = pd.read_csv(cases_path)
    print(f'Loaded {len(cases)} failure cases.')
else:
    cases = pd.DataFrame()
    print(f'No failure_cases.csv found at {cases_path}.')
    print('Run: python -m analysis export-failures --results ... --out Analysis/output/failures')

## 1. Failure Dataset Scope

In [ ]:
if not cases.empty:
    print(f'Total failure cases: {len(cases)}')
    if 'lane' in cases.columns:
        print('By lane:')
        print(cases['lane'].value_counts())
    if 'repo_name' in cases.columns:
        print('\nRepositories in failure set:')
        print(cases['repo_name'].value_counts())

## 2. Preliminary Machine Labels

In [ ]:
if not cases.empty and 'preliminary_failure_label' in cases.columns:
    label_counts = cases['preliminary_failure_label'].value_counts()
    print('Preliminary label distribution:')
    print(label_counts)

    # Flag labels not in schema
    unexpected = set(label_counts.index) - set(PRELIMINARY_FAILURE_LABELS)
    if unexpected:
        print(f'\nUnexpected labels (not in schema): {unexpected}')

## 3. Failure Category Counts

In [ ]:
if not cases.empty and 'preliminary_failure_label' in cases.columns:
    fig, ax = plt.subplots(figsize=(12, 5))
    failure_category_bar(cases, lane_col='lane', category_col='preliminary_failure_label', ax=ax)
    plt.tight_layout()
    plt.show()

    if 'tool_id' in cases.columns:
        fig, ax = plt.subplots(figsize=(12, 5))
        failure_category_bar(cases, lane_col='tool_id', category_col='preliminary_failure_label', ax=ax)
        ax.set_title('Failure categories by tool')
        plt.tight_layout()
        plt.show()

## 4. Stratified Sampling Plan

In [ ]:
from analysis.export_failure_cases import sample_cases

if not cases.empty:
    # Stratified by label: 5 per label
    stratified = sample_cases(cases, strategy='stratified-label', top_n=5)
    print(f'Stratified sample size: {len(stratified)}')
    if 'preliminary_failure_label' in stratified.columns:
        print(stratified['preliminary_failure_label'].value_counts())

## 5. Paired Disagreement Cases

In [ ]:
# Cases where one lane failed and the other succeeded for the same candidate
if not cases.empty and 'candidate_key' in cases.columns and 'lane' in cases.columns:
    attempts_path = DATA_DIR.parent / 'evaluation_candidates.csv'
    if attempts_path.exists():
        cands = pd.read_csv(attempts_path)
        from analysis.normalize import build_paired_comparison
        paired = build_paired_comparison(cands)
        disagreements = paired[paired['winner'].isin(['llm_won', 'agentic_won'])]
        print(f'Disagreement pairs: {len(disagreements)}')
        display(disagreements[['candidate_key', 'winner', 'llm_validated_success', 'agentic_validated_success']].head(10))
    else:
        print('evaluation_candidates.csv not found.')

## 6. High-Severity Failures

In [ ]:
from analysis.export_failure_cases import sample_cases

if not cases.empty:
    severe = sample_cases(cases, strategy='high-severity')
    print(f'High-severity cases: {len(severe)}')
    cols = [c for c in ['failure_case_id', 'lane', 'tool_id', 'preliminary_failure_label',
                         'source_method_name', 'outcome_summary'] if c in severe.columns]
    display(severe[cols].head(20))

## 7. Exported Case Files

In [ ]:
md_dir = DATA_DIR / 'failure_cases'
if md_dir.exists():
    md_files = list(md_dir.glob('*.md'))
    print(f'Markdown case files: {len(md_files)}')
    if md_files:
        print('Sample:')
        print(md_files[0].read_text()[:800])
else:
    print('No markdown case files found.')
    print('Re-run export with: python -m analysis export-failures ... --markdown')